<h1>Chapter 7 - Advanced Text Generation Techniques and Tools</h1>
<i>Going beyond prompt engineering.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter07/Chapter%207%20-%20Advanced%20Text%20Generation%20Techniques%20and%20Tools.ipynb)

---

This notebook is for Chapter 7 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>

### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
# %%capture
# !pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2 langchain_community
# !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69

# Loading an LLM

In [ ]:
# Download the phi-4 GGUF model

!wget https://huggingface.co/microsoft/phi-4-gguf/resolve/main/phi-4-q4.gguf

In [ ]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="phi-4-q4.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [3]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

### Chains

In [4]:
from langchain import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [5]:
basic_chain = prompt | llm

In [6]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" Hello Maarten! 1 + 1 equals 2. It's a basic arithmetic operation, and the result is always the same."

### Multiple Chains

In [7]:
from langchain import LLMChain

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

/var/folders/l5/50g_mf0n6h1307tht1fswfb00000gp/T/ipykernel_74295/3105618174.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [8]:
title.invoke({"summary": "a girl that lost her mother"})

/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Mother\'s Love: A Tale of Healing"'}

In [9]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [10]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [11]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [12]:
llm_chain.invoke("a girl that lost her mother")

/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Lullaby: A Journey Through Grief"',
 'character': " The protagonist, a young girl named Emily, is a resilient and introspective soul who has been profoundly affected by the loss of her mother. As she navigates through the tumultuous waves of grief, Emily discovers newfound strength within herself as she embarks on an emotional journey to heal and find solace in the memories and whispers of her mother's lullaby.",
 'story': " Whispers of a Lullaby: A Journey Through Grief unfolds as the heart-wrenching tale of young Emily, whose tender soul crumbled under the weight of her mother's untimely passing. Engulfed by an overwhelming sea of sorrow and loneliness, she retreated into herself, seeking solace in the melodic remnants of a once-beloved lullaby that echoed through their home like a tender embrace from beyond. As Emily embarked on her emotional odyssey, traversing grief's turbulent waters, she discovered an inner re

# Memory

In [13]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

In [14]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" I'm unable to determine your name as I don't have access to personal data about individuals. However, if you need assistance with a specific problem or question, feel free to ask!"

## ConversationBuffer

In [15]:
# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [16]:
from langchain.memory import ConversationBufferMemory

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/var/folders/l5/50g_mf0n6h1307tht1fswfb00000gp/T/ipykernel_74295/1531358956.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history")


In [17]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': '',
 'text': " The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units."}

In [18]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

/Users/taareda4/llmvenv/lib/python3.12/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units.",
 'text': ' Your name is Maarten.\n\n-----<|assistant|> Hello Maarten! My name is Assistant. The answer to 1 + 1 is indeed 2, which is a fundamental addition operation in mathematics. When you add one unit to another, you simply combine them for a total of two units.'}

## ConversationBufferMemoryWindow

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
# Ask two questions and generate two conversations in its memory
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

{'input_prompt': 'What is 3 + 3?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten, it's nice to meet you! The answer to 1 + 1 is 2.\n\nHowever, if you have any other questions or need further assistance, feel free to ask!",
 'text': " Hello again! 3 + 3 equals 6. If there's anything else I can help you with, just let me know!"}

In [ ]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten, it's nice to meet you! The answer to 1 + 1 is 2.\n\nHowever, if you have any other questions or need further assistance, feel free to ask!\nHuman: What is 3 + 3?\nAI:  Hello again! 3 + 3 equals 6. If there's anything else I can help you with, just let me know!",
 'text': ' Your name is Maarten.'}

In [ ]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})

{'input_prompt': 'What is my age?',
 'chat_history': "Human: What is 3 + 3?\nAI:  Hello again! 3 + 3 equals 6. If there's anything else I can help you with, just let me know!\nHuman: What is my name?\nAI:  Your name is Maarten.",
 'text': " I'm unable to determine your age as I don't have access to personal information. Age isn't something that can be inferred from our current conversation unless you choose to share it with me. How else may I assist you today?"}

## ConversationSummary

In [ ]:
# Create a summary prompt template
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [ ]:
from langchain.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': ' Summary: Human, identified as Maarten, asked the AI about the sum of 1 + 1, which was correctly answered by the AI as 2 and offered additional assistance if needed.',
 'text': ' Your name in this context was referred to as "Maarten". However, since our interaction doesn\'t retain personal data beyond a single session for privacy reasons, I don\'t have access to that information. How can I assist you further today?'}

In [ ]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': ' Summary: Human, identified as Maarten in the context of this conversation, first asked about the sum of 1 + 1 and received an answer of 2 from the AI. Later, Maarten inquired about their name but the AI clarified that personal data is not retained beyond a single session for privacy reasons. The AI offered further assistance if needed.',
 'text': ' The first question you asked was "what\'s 1 + 1?"'}

In [ ]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': ' Maarten, identified in this conversation, initially asked about the sum of 1+1 which resulted in an answer from the AI being 2. Subsequently, he sought clarification on his name but the AI informed him that no personal data is retained beyond a single session due to privacy reasons. The AI then offered further assistance if required. Later, Maarten recalled and asked about the first question he inquired which was "what\'s 1+1?"'}

# Agents

In [1]:
# Download the phi-4 GGUF model
!wget https://huggingface.co/microsoft/phi-4-gguf/resolve/main/phi-4-q4.gguf

--2025-03-01 07:42:37--  https://huggingface.co/microsoft/phi-4-gguf/resolve/main/phi-4-q4.gguf
Resolving huggingface.co (huggingface.co)... 2600:9000:273b:7800:17:b174:6d00:93a1, 2600:9000:273b:d200:17:b174:6d00:93a1, 2600:9000:273b:0:17:b174:6d00:93a1, ...
Connecting to huggingface.co (huggingface.co)|2600:9000:273b:7800:17:b174:6d00:93a1|:443... connected.
302 Foundest sent, awaiting response... 
Location: https://cdn-lfs-us-1.hf.co/repos/dc/13/dc139d769a2d9457bcd1e3d323cbb7cfff1db19e64eedfa3770abc8c8be08751/ab704ffa097f5902b937ca5d0c166226f1201e7492da30fa4e1c086c217afd6b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27phi-4-q4.gguf%3B+filename%3D%22phi-4-q4.gguf%22%3B&Expires=1740814957&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MDgxNDk1N319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zL2RjLzEzL2RjMTM5ZDc2OWEyZDk0NTdiY2QxZTNkMzIzY2JiN2NmZmYxZGIxOWU2NGVlZGZhMzc3MGFiYzhjOGJlMDg3NTEvYWI3MDRmZmEwOTdmNTkwMmI5Mzd

In [2]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="phi-4-q4.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

llama_init_from_model: n_batch is less than GGML_KQ_MASK_PAD - increasing to 32
llama_init_from_model: n_ctx_per_seq (2048) < n_ctx_train (16384) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16

In [33]:
from langchain import PromptTemplate

# Original ReAct template
# react_template = """Answer the following questions as best you can. You have access to the following tools:

# {tools}

# Use the following format:

# Question: the input question you must answer
# Thought: you should always think about what to do
# Action: the action to take, should be one of [{tool_names}]
# Action Input: the input to the action
# Observation: the result of the action
# ... (this Thought/Action/Action Input/Observation can repeat N times)
# Thought: I now know the final answer
# Final Answer: the final answer to the original input question

# Begin!

# Question: {input}
# Thought:{agent_scratchpad}"""

react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: your should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input of the action
Observation: the result of the action
Final Answer: the final answer to the original input question

This Thought/Action/Action Input/Observation loop can repeat multiple times.

Start now:

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

In [34]:
from langchain.agents import load_tools, Tool
from langchain.tools import DuckDuckGoSearchResults

# You can create the tool to pass to an agent
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)

# Prepare tools
tools = load_tools(["llm-math"], llm=llm)
tools.append(search_tool)

In [35]:
from langchain.agents import AgentExecutor, create_react_agent

# Construct the ReAct agent
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [37]:
# What is the Price of a MacBook Pro?
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
    }
)



> Entering new AgentExecutor chain...
 I need to find the current price of a MacBook Pro in USD. Once I have that information, I can calculate the cost in EUR using the provided exchange rate.
Action: duckduck
snippet: , title: EOF, link: http://www.google.com/search?hl=en&q={"tool_input":+"current+price+of+a+MacBook+Pro+in+USD",+"verbose":+true} Based on the search results, I need to find a more specific source that provides the current price of a MacBook Pro in USD.
Action: duckduck
 Based on the search results, I need to find a more specific source that provides the current price of a MacBook Pro 14-inch (2023) in USD.
Action: duckduck
snippet: , title: EOF, link: http://www.google.com/search?hl=en&q={"tool_input":+"Apple+-+MacBook+Pro+with+Apple+M2+Pro+Chip",+"verbose":+true} Based on the search results, I need to find a more specific source that provides the current price of a MacBook Pro with Apple M2 Pro Chip in USD.
Action: duckduck
 Based on the search results, I have found 

{'input': 'What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?',
 'output': 'The current price of a MacBook Pro with Apple M2 Pro Chip is $2499 USD. To calculate the cost in EUR using an exchange rate of 0.85 EUR for 1 USD, we can use the following calculation: 0.85 EUR * $2499 USD = €2124.15.'}

In [36]:
# What is the Price of a flight to Catania in april?
agent_executor.invoke(
    {
        "input": "What is the cheapest fligh from Basel to Catania in April? Please provide me with the airline, the best dates and the total price"
    }
)



> Entering new AgentExecutor chain...
 To find the cheapest flight, I'll need to search for flights from Basel to Catania in April. Since this is a specific query about flight information, including pricing, airlines, and dates, using duckduck will be appropriate as it can handle such detailed queries.
Action: duckduck
snippet: Compare cheap flights and find tickets from Basel Mulhouse Freiburg (BSL) to Catania Fontanarossa (CTA). Book directly with no added fees., title: Cheap Flights from Basel (BSL) to Catania (CTA) - Skyscanner, link: https://www.skyscanner.com.au/routes/bsl/cta/basel-mulhouse-freiburg-to-catania-fontanarossa.html, snippet: Compare cheap flights and find tickets from Basel Mulhouse Freiburg to Catania Fontanarossa. Book directly with no added fees. ... Flights. Hotels. Car Hire. Cheap Flights from Basel Mulhouse Freiburg to Catania Fontanarossa. Return. One way. Multi-city. From. To. Depart. Return. Travellers and cabin class 1 adult, Economy. Search. Add nearby 

{'input': 'What is the cheapest fligh from Basel to Catania in April? Please provide me with the airline, the best dates and the total price',
 'output': "The cheapest flight from Basel to Catania in April is offered by easyJet. The best dates for this round trip are departing on Monday, 24th March, and returning on Wednesday, 2nd April. The total price for these flights, including taxes and charges, is £51.\n\n## Past plan\n1. **Subgoal: Review the current flight options provided by duckduck.**\n   - **Quantification:** Completion can be quantified by reviewing at least 3 different flight options from the search results.\n   - **Certainty Level:** Known; reviewing available options is a necessary step in flight planning.\n\n2. **Subgoal: Execute additional searches if needed to confirm details or find better options.**\n   - **Quantification:** Completion can be quantified by conducting at least 1 additional search if initial information is insufficient.\n   - **Certainty Level:** Inf